**12-month behavioral features**


In [11]:
#!/usr/bin/env python3

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path(r"C:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

customer_info = pd.read_csv(RAW_DATA_DIR / "customer_info.csv")
usage_data = pd.read_csv(RAW_DATA_DIR / "usage_data.csv")
churn_labels = pd.read_csv(RAW_DATA_DIR / "churn_labels.csv")

# Convert date columns to datetime format
usage_data["Month"] = pd.to_datetime(
    usage_data["Month"],
    errors="coerce"                       #errors="coerce" will convert invalid parsing to NaT
)

customer_info["SignupDate"] = pd.to_datetime(
    customer_info["SignupDate"],
    errors="coerce"                       #errors="coerce" will convert invalid parsing to NaT
)


print("customer_info:", customer_info.shape)
print("usage_data:", usage_data.shape)
print("churn_labels:", churn_labels.shape)

customer_info: (10200, 7)
usage_data: (120000, 6)
churn_labels: (10000, 2)


In [12]:
# Remove exact duplicate customer rows
customer_info_clean = customer_info.drop_duplicates().copy()        #copy() is used to create a new DataFrame to avoid SettingWithCopyWarning

# For remaining duplicated CustomerIDs, keep earliest SignupDate
customer_info_clean = (
    customer_info_clean
    .sort_values(["CustomerID", "SignupDate"])       
    .drop_duplicates(
        subset="CustomerID",                #subset="CustomerID" specifies that we want to drop duplicates based on the CustomerID column
        keep="first"
    )
    .reset_index(drop=True)                   #drop=True is used to avoid adding the old index as a column in the new DataFrame
)

# Age missingness flag
customer_info_clean["AgeMissing"] = (
    customer_info_clean["Age"]
    .isna()
    .astype(int)
)

# Invalid MonthlyCharges flag
customer_info_clean["MonthlyChargesInvalid"] = (
    customer_info_clean["MonthlyCharges"] <= 0
).astype(int)

# Replace invalid charges with missing values
customer_info_clean.loc[
    customer_info_clean["MonthlyCharges"] <= 0,             
    "MonthlyCharges"
] = np.nan

print("Clean customer shape:", customer_info_clean.shape)
print("Unique customers:", customer_info_clean["CustomerID"].nunique())

Clean customer shape: (10000, 9)
Unique customers: 10000


In [13]:
# Calculate usage features for each customer
usage_features = (
    usage_data
    .groupby("CustomerID")
    .agg(
        call_avg=("CallMinutes", "mean"),
        call_std=("CallMinutes", "std"),
        call_min=("CallMinutes", "min"),
        call_max=("CallMinutes", "max"),

        data_avg=("DataUsageGB", "mean"),
        data_std=("DataUsageGB", "std"),
        data_min=("DataUsageGB", "min"),
        data_max=("DataUsageGB", "max"),

        sms_avg=("SMSCount", "mean"),
        sms_std=("SMSCount", "std"),

        complaints_total=("Complaints", "sum"),
        complaints_avg=("Complaints", "mean"),
        complaints_max=("Complaints", "max")
    )
    .reset_index()
)

print(usage_features.shape)
display(usage_features.head())

(10000, 14)


,CustomerID,call_avg,call_std,call_min,call_max,data_avg,data_std,data_min,data_max,sms_avg,sms_std,complaints_total,complaints_avg,complaints_max
0,CUST_0,492.233333,52.734272,434.82,590.26,9.795833,2.734661,5.99,14.16,25.000000,12.202831,7,0.583333,2
1,CUST_1,452.007500,47.582323,396.88,564.07,10.298333,4.197458,3.45,17.17,34.583333,13.013687,13,1.083333,3
2,CUST_10,628.351667,39.321456,534.43,681.10,8.602500,4.848668,0.00,14.72,23.083333,13.446583,12,1.000000,3
3,CUST_100,534.539167,47.373632,433.72,582.60,9.587500,3.615105,5.43,14.55,29.000000,12.037819,5,0.416667,1
4,CUST_1000,493.881667,52.645219,413.70,600.54,10.914167,2.333579,6.39,13.90,26.250000,15.040098,12,1.000000,2


In [14]:
# Calculate usage features for the most recent 3 months
recent_3m = (
    usage_data[
        usage_data["Month"].dt.month >= 10
    ]
    .groupby("CustomerID")
    .agg(
        call_avg_last3m=("CallMinutes", "mean"),
        data_avg_last3m=("DataUsageGB", "mean"),
        sms_avg_last3m=("SMSCount", "mean"),
        complaints_total_last3m=("Complaints", "sum")
    )
    .reset_index()
)

print(recent_3m.shape)
display(recent_3m.head())

(10000, 5)


,CustomerID,call_avg_last3m,data_avg_last3m,sms_avg_last3m,complaints_total_last3m
0,CUST_0,488.696667,11.326667,22.000000,1
1,CUST_1,436.446667,10.943333,42.333333,5
2,CUST_10,661.150000,9.740000,37.333333,1
3,CUST_100,549.700000,10.980000,24.666667,0
4,CUST_1000,457.233333,11.360000,41.666667,2


In [15]:
# Calculate usage features for the first 3 months

first_3m = (
    usage_data[
        usage_data["Month"].dt.month <= 3
    ]
    .groupby("CustomerID")
    .agg(
        call_avg_first3m=("CallMinutes", "mean"),
        data_avg_first3m=("DataUsageGB", "mean"),
        sms_avg_first3m=("SMSCount", "mean"),
        complaints_total_first3m=("Complaints", "sum")
    )
    .reset_index()
)

print(first_3m.shape)
display(first_3m.head())

(10000, 5)


,CustomerID,call_avg_first3m,data_avg_first3m,sms_avg_first3m,complaints_total_first3m
0,CUST_0,500.990000,9.120000,19.333333,3
1,CUST_1,463.056667,10.783333,31.333333,4
2,CUST_10,645.703333,5.626667,22.000000,3
3,CUST_100,521.390000,7.723333,25.666667,2
4,CUST_1000,520.606667,10.183333,31.000000,2


In [16]:
# Merge usage features with recent and first 3 months features
usage_features = (
    usage_features
    .merge(
        recent_3m,
        on="CustomerID",
        how="left"
    )
    .merge(
        first_3m,
        on="CustomerID",
        how="left"
    )
)

print(usage_features.shape)
display(usage_features.head())

(10000, 22)


,CustomerID,call_avg,call_std,call_min,call_max,data_avg,data_std,data_min,data_max,sms_avg,sms_std,complaints_total,complaints_avg,complaints_max,call_avg_last3m,data_avg_last3m,sms_avg_last3m,complaints_total_last3m,call_avg_first3m,data_avg_first3m,sms_avg_first3m,complaints_total_first3m
0,CUST_0,492.233333,52.734272,434.82,590.26,9.795833,2.734661,5.99,14.16,25.000000,12.202831,7,0.583333,2,488.696667,11.326667,22.000000,1,500.990000,9.120000,19.333333,3
1,CUST_1,452.007500,47.582323,396.88,564.07,10.298333,4.197458,3.45,17.17,34.583333,13.013687,13,1.083333,3,436.446667,10.943333,42.333333,5,463.056667,10.783333,31.333333,4
2,CUST_10,628.351667,39.321456,534.43,681.10,8.602500,4.848668,0.00,14.72,23.083333,13.446583,12,1.000000,3,661.150000,9.740000,37.333333,1,645.703333,5.626667,22.000000,3
3,CUST_100,534.539167,47.373632,433.72,582.60,9.587500,3.615105,5.43,14.55,29.000000,12.037819,5,0.416667,1,549.700000,10.980000,24.666667,0,521.390000,7.723333,25.666667,2
4,CUST_1000,493.881667,52.645219,413.70,600.54,10.914167,2.333579,6.39,13.90,26.250000,15.040098,12,1.000000,2,457.233333,11.360000,41.666667,2,520.606667,10.183333,31.000000,2


In [17]:
# Calculate change in usage behavior for each customer
usage_features["call_change_3m"] = (
    usage_features["call_avg_last3m"]
    - usage_features["call_avg_first3m"]
)

usage_features["data_change_3m"] = (
    usage_features["data_avg_last3m"]
    - usage_features["data_avg_first3m"]
)

usage_features["sms_change_3m"] = (
    usage_features["sms_avg_last3m"]
    - usage_features["sms_avg_first3m"]
)

usage_features["complaints_change_3m"] = (
    usage_features["complaints_total_last3m"]
    - usage_features["complaints_total_first3m"]
)


print(usage_features.shape)
display(usage_features.head())

(10000, 26)


,CustomerID,call_avg,call_std,call_min,call_max,data_avg,data_std,data_min,data_max,sms_avg,sms_std,complaints_total,complaints_avg,complaints_max,call_avg_last3m,data_avg_last3m,sms_avg_last3m,complaints_total_last3m,call_avg_first3m,data_avg_first3m,sms_avg_first3m,complaints_total_first3m,call_change_3m,data_change_3m,sms_change_3m,complaints_change_3m
0,CUST_0,492.233333,52.734272,434.82,590.26,9.795833,2.734661,5.99,14.16,25.000000,12.202831,7,0.583333,2,488.696667,11.326667,22.000000,1,500.990000,9.120000,19.333333,3,-12.293333,2.206667,2.666667,-2
1,CUST_1,452.007500,47.582323,396.88,564.07,10.298333,4.197458,3.45,17.17,34.583333,13.013687,13,1.083333,3,436.446667,10.943333,42.333333,5,463.056667,10.783333,31.333333,4,-26.610000,0.160000,11.000000,1
2,CUST_10,628.351667,39.321456,534.43,681.10,8.602500,4.848668,0.00,14.72,23.083333,13.446583,12,1.000000,3,661.150000,9.740000,37.333333,1,645.703333,5.626667,22.000000,3,15.446667,4.113333,15.333333,-2
3,CUST_100,534.539167,47.373632,433.72,582.60,9.587500,3.615105,5.43,14.55,29.000000,12.037819,5,0.416667,1,549.700000,10.980000,24.666667,0,521.390000,7.723333,25.666667,2,28.310000,3.256667,-1.000000,-2
4,CUST_1000,493.881667,52.645219,413.70,600.54,10.914167,2.333579,6.39,13.90,26.250000,15.040098,12,1.000000,2,457.233333,11.360000,41.666667,2,520.606667,10.183333,31.000000,2,-63.373333,1.176667,10.666667,0


In [18]:
usage_data["MonthNumber"] = usage_data["Month"].dt.month

display(
    usage_data[
        ["Month", "MonthNumber"]
    ]
    .drop_duplicates()
    .sort_values("MonthNumber")
)

,Month,MonthNumber
0,2023-01-31,1
1,2023-02-28,2
2,2023-03-31,3
3,2023-04-30,4
4,2023-05-31,5
5,2023-06-30,6
6,2023-07-31,7
7,2023-08-31,8
8,2023-09-30,9
9,2023-10-31,10


In [19]:
# Calculate slope of usage features over time for each customer

def calculate_slope(group, value_column):
    x = group["MonthNumber"].values
    y = group[value_column].values

    slope = np.polyfit(x, y, 1)[0]

    return slope

call_slopes = (
    usage_data
    .groupby("CustomerID")
    .apply(
        lambda group: calculate_slope(
            group,
            "CallMinutes"
        ),
        include_groups=False
    )
    .reset_index(name="call_trend_slope")
)

data_slopes = (
    usage_data
    .groupby("CustomerID")
    .apply(
        lambda group: calculate_slope(
            group,
            "DataUsageGB"
        ),
        include_groups=False
    )
    .reset_index(name="data_trend_slope")
)

complaint_slopes = (
    usage_data
    .groupby("CustomerID")
    .apply(
        lambda group: calculate_slope(
            group,
            "Complaints"
        ),
        include_groups=False
    )
    .reset_index(name="complaints_trend_slope")
)

In [20]:
usage_features = (
    usage_features
    .merge(
        call_slopes,
        on="CustomerID",
        how="left"
    )
    .merge(
        data_slopes,
        on="CustomerID",
        how="left"
    )
    .merge(
        complaint_slopes,
        on="CustomerID",
        how="left"
    )
)

print(usage_features.shape)

display(
    usage_features[
        [
            "CustomerID",
            "call_trend_slope",
            "data_trend_slope",
            "complaints_trend_slope"
        ]
    ].head()
)


(10000, 29)


,CustomerID,call_trend_slope,data_trend_slope,complaints_trend_slope
0,CUST_0,0.252378,0.219266,-9.440559e-02
1,CUST_1,-2.704161,-0.147063,1.748252e-02
2,CUST_10,1.261119,0.260524,-5.594406e-02
3,CUST_100,2.170874,0.341503,-7.342657e-02
4,CUST_1000,-8.303916,0.146958,-1.531626e-17


In [21]:
usage_features.shape

(10000, 29)

In [22]:
# Combine all features into a single modeling dataset
model_data = (
    customer_info_clean
    .merge(
        usage_features,
        on="CustomerID",
        how="inner"
    )
    .merge(
        churn_labels,
        on="CustomerID",
        how="inner"
    )
)

print("Modeling dataset shape:", model_data.shape)
display(model_data.head())

Modeling dataset shape: (10000, 38)


,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,AgeMissing,MonthlyChargesInvalid,call_avg,call_std,call_min,call_max,data_avg,data_std,data_min,data_max,sms_avg,sms_std,complaints_total,complaints_avg,complaints_max,call_avg_last3m,data_avg_last3m,sms_avg_last3m,complaints_total_last3m,call_avg_first3m,data_avg_first3m,sms_avg_first3m,complaints_total_first3m,call_change_3m,data_change_3m,sms_change_3m,complaints_change_3m,call_trend_slope,data_trend_slope,complaints_trend_slope,Churn
0,CUST_0,NaN,Male,Urban,Yearly,48.18,2021-03-20,1,0,492.233333,52.734272,434.82,590.26,9.795833,2.734661,5.99,14.16,25.000000,12.202831,7,0.583333,2,488.696667,11.326667,22.000000,1,500.990000,9.120000,19.333333,3,-12.293333,2.206667,2.666667,-2,0.252378,0.219266,-9.440559e-02,0
1,CUST_1,NaN,Female,Suburban,Prepaid,38.94,2020-01-25,1,0,452.007500,47.582323,396.88,564.07,10.298333,4.197458,3.45,17.17,34.583333,13.013687,13,1.083333,3,436.446667,10.943333,42.333333,5,463.056667,10.783333,31.333333,4,-26.610000,0.160000,11.000000,1,-2.704161,-0.147063,1.748252e-02,0
2,CUST_10,NaN,Female,Urban,Prepaid,47.81,2022-04-29,1,0,628.351667,39.321456,534.43,681.10,8.602500,4.848668,0.00,14.72,23.083333,13.446583,12,1.000000,3,661.150000,9.740000,37.333333,1,645.703333,5.626667,22.000000,3,15.446667,4.113333,15.333333,-2,1.261119,0.260524,-5.594406e-02,0
3,CUST_100,62.0,Female,Urban,Monthly,29.80,2021-10-14,0,0,534.539167,47.373632,433.72,582.60,9.587500,3.615105,5.43,14.55,29.000000,12.037819,5,0.416667,1,549.700000,10.980000,24.666667,0,521.390000,7.723333,25.666667,2,28.310000,3.256667,-1.000000,-2,2.170874,0.341503,-7.342657e-02,0
4,CUST_1000,NaN,Female,Suburban,Monthly,61.62,2018-06-10,1,0,493.881667,52.645219,413.70,600.54,10.914167,2.333579,6.39,13.90,26.250000,15.040098,12,1.000000,2,457.233333,11.360000,41.666667,2,520.606667,10.183333,31.000000,2,-63.373333,1.176667,10.666667,0,-8.303916,0.146958,-1.531626e-17,0


In [23]:
#Inspect the modeling dataset

print("Rows:", len(model_data))
print("Unique customers:", model_data["CustomerID"].nunique())
print("Duplicate customer IDs:", model_data["CustomerID"].duplicated().sum())

print("\nTarget distribution:")
print(model_data["Churn"].value_counts())

print("\nTarget percentage:")
print(
    model_data["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Rows: 10000
Unique customers: 10000
Duplicate customer IDs: 0

Target distribution:
Churn
0    9214
1     786
Name: count, dtype: int64

Target percentage:
Churn
0    92.14
1     7.86
Name: proportion, dtype: float64


In [24]:
#Inspect missing values in the modeling dataset
missing_summary = (
    model_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[
    missing_summary > 0
]

display(missing_summary)

Age               3500
MonthlyCharges       5
dtype: int64

In [25]:
model_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 38 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   CustomerID                10000 non-null  str           
 1   Age                       6500 non-null   float64       
 2   Gender                    10000 non-null  str           
 3   Region                    10000 non-null  str           
 4   ContractType              10000 non-null  str           
 5   MonthlyCharges            9995 non-null   float64       
 6   SignupDate                10000 non-null  datetime64[us]
 7   AgeMissing                10000 non-null  int64         
 8   MonthlyChargesInvalid     10000 non-null  int64         
 9   call_avg                  10000 non-null  float64       
 10  call_std                  10000 non-null  float64       
 11  call_min                  10000 non-null  float64       
 12  call_max                  1000

**identify suspicious signup dates**

In [26]:
# First observed usage month for each customer
first_observed_usage = (
    usage_data
    .groupby("CustomerID")["Month"]
    .min()
    .reset_index()
    .rename(
        columns={
            "Month": "FirstObservedUsageMonth"
        }
    )
)

model_data = model_data.merge(
    first_observed_usage,
    on="CustomerID",
    how="left"
)

In [28]:
# Flag customers with invalid signup dates (after first observed usage month)

model_data["SignupDateInvalid"] = (
    model_data["SignupDate"]
    > model_data["FirstObservedUsageMonth"]
).astype(int)

print(
    model_data["SignupDateInvalid"]
    .value_counts()
)

SignupDateInvalid
0    9249
1     751
Name: count, dtype: int64


In [29]:
# Reference date for feature engineering
REFERENCE_DATE = pd.Timestamp("2023-12-31")

# Calculate tenure in months for each customer
model_data["TenureMonths"] = (
    (
        REFERENCE_DATE
        - model_data["SignupDate"]
    ).dt.days / 30.44
)

model_data["TenureMonths"].describe()

count    10000.000000
mean        39.126534
std         19.203105
min          1.905388
25%         22.339028
50%         39.093298
75%         55.814717
max         71.944809
Name: TenureMonths, dtype: float64

In [30]:
# Flag customers with invalid signup dates (after first observed usage month) and set TenureMonths to NaN for those customers
model_data.loc[
    model_data["SignupDateInvalid"] == 1,
    "TenureMonths"
] = np.nan

print(
    "Missing TenureMonths:",
    model_data["TenureMonths"].isna().sum()
)

print(
    "Invalid signup dates:",
    model_data["SignupDateInvalid"].sum()
)

Missing TenureMonths: 751
Invalid signup dates: 751


In [31]:
missing_summary = (
    model_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary[
        missing_summary > 0
    ]
)

Age               3500
TenureMonths       751
MonthlyCharges       5
dtype: int64

In [32]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    PROCESSED_DATA_DIR
    / "customer_churn_features.csv"
)

model_data.to_csv(
    output_path,
    index=False
)

print("Saved to:", output_path)
print("Shape:", model_data.shape)

Saved to: C:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system\data\processed\customer_churn_features.csv
Shape: (10000, 41)


In [33]:
saved_data = pd.read_csv(output_path)

print("Reloaded shape:", saved_data.shape)
display(saved_data.head())

Reloaded shape: (10000, 41)


,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,AgeMissing,MonthlyChargesInvalid,call_avg,call_std,call_min,call_max,data_avg,data_std,data_min,data_max,sms_avg,sms_std,complaints_total,complaints_avg,complaints_max,call_avg_last3m,data_avg_last3m,sms_avg_last3m,complaints_total_last3m,call_avg_first3m,data_avg_first3m,sms_avg_first3m,complaints_total_first3m,call_change_3m,data_change_3m,sms_change_3m,complaints_change_3m,call_trend_slope,data_trend_slope,complaints_trend_slope,Churn,FirstObservedUsageMonth,SignupDateInvalid,TenureMonths
0,CUST_0,NaN,Male,Urban,Yearly,48.18,2021-03-20,1,0,492.233333,52.734272,434.82,590.26,9.795833,2.734661,5.99,14.16,25.000000,12.202831,7,0.583333,2,488.696667,11.326667,22.000000,1,500.990000,9.120000,19.333333,3,-12.293333,2.206667,2.666667,-2,0.252378,0.219266,-9.440559e-02,0,2023-01-31,0,33.377135
1,CUST_1,NaN,Female,Suburban,Prepaid,38.94,2020-01-25,1,0,452.007500,47.582323,396.88,564.07,10.298333,4.197458,3.45,17.17,34.583333,13.013687,13,1.083333,3,436.446667,10.943333,42.333333,5,463.056667,10.783333,31.333333,4,-26.610000,0.160000,11.000000,1,-2.704161,-0.147063,1.748252e-02,0,2023-01-31,0,47.174770
2,CUST_10,NaN,Female,Urban,Prepaid,47.81,2022-04-29,1,0,628.351667,39.321456,534.43,681.10,8.602500,4.848668,0.00,14.72,23.083333,13.446583,12,1.000000,3,661.150000,9.740000,37.333333,1,645.703333,5.626667,22.000000,3,15.446667,4.113333,15.333333,-2,1.261119,0.260524,-5.594406e-02,0,2023-01-31,0,20.072273
3,CUST_100,62.0,Female,Urban,Monthly,29.80,2021-10-14,0,0,534.539167,47.373632,433.72,582.60,9.587500,3.615105,5.43,14.55,29.000000,12.037819,5,0.416667,1,549.700000,10.980000,24.666667,0,521.390000,7.723333,25.666667,2,28.310000,3.256667,-1.000000,-2,2.170874,0.341503,-7.342657e-02,0,2023-01-31,0,26.544021
4,CUST_1000,NaN,Female,Suburban,Monthly,61.62,2018-06-10,1,0,493.881667,52.645219,413.70,600.54,10.914167,2.333579,6.39,13.90,26.250000,15.040098,12,1.000000,2,457.233333,11.360000,41.666667,2,520.606667,10.183333,31.000000,2,-63.373333,1.176667,10.666667,0,-8.303916,0.146958,-1.531626e-17,0,2023-01-31,0,66.688568
